# Evaluating an LLM in your language

The very first step is to change your runtime!

In the toolbar at the top of this window, go to `Runtime` `>` `Change runtime type`.

Select `G4 GPU`. Then you can begin.

## First, a full example with a sample test set

### Install packages

In [ ]:
!pip install transformers
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.7 MB/s eta 0:00:00


### Load Gemma model

A model small enough to fit in this runtime

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

model.cuda() # <-- This line will cause an error if you haven't changed
#                  your runtime to a GPU runtime.

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

#### Running the model

This shows how inference is run with the model.

**Note** that this cell only takes a few seconds to run. That is thanks to the GPU. If you were on a CPU runtime and/or hadn't specified `model.cuda()` above, this cell would take a full minute to run.

In [ ]:
# Prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short joke about saving RAM."},
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

Here are a few options for a short joke about saving RAM, depending on the style you prefer:

**Option 1 (Relatable Tech Frustration):**

> Why did the computer break up with the multitasking wizard? Because he kept overloading his RAM!

**Option 2 (Short & Punchy):**

> My computer is running so slow, I think it's just politely asking me to save some RAM.

**Option 3 (A Bit More Punny):**

> I tried to multitask all day, but my RAM just sighed and said, "I'm just trying to keep my cool."

**Option 4 (Self-Deprecating):**

> I'm not lazy, I'm just aggressively trying to save RAM so my computer doesn't throw a tantrum.

---

**Which style do you like best?** 😊


Let's try it with a batched input.

Again, this runs in a few seconds. It would take 8 minutes without a GPU.

In [ ]:
# Prompt
messages = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short joke about saving RAM."},
    ],
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short joke about saving money."},
    ],
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short joke about saving time."},
    ],
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
responses = [
    processor.decode(
        output[input_len:], skip_special_tokens=True
                     ) for output in outputs
]

for response in responses:
    print(response)
    print()
    print('-' * 10)
    print()

Here are a few options, depending on the style you prefer:

**Option 1 (Relatable Tech Struggle):**

> Why did the computer break up with the gamer? Because he kept hogging all the RAM!

**Option 2 (Short & Punchy):**

> My computer told me to free up some RAM. I said, "Not until I finish this one tab!"

**Option 3 (A Little More Observational):**

> Saving RAM is like trying to fit a whole library into a shoebox. It's ambitious, and usually results in a crash.

**Which one do you like best?** 😊
----------

Here are a few options, depending on your style:

**Option 1 (Relatable):**

> I started saving money. Now I have so much in the bank, I'm starting to develop a relationship with my savings account... and it's very clingy.

**Option 2 (Short & Punchy):**

> Why did the penny break up with the dollar? They just couldn't agree on their future!

**Option 3 (A bit of self-deprecating):**

> My financial goal is to save enough money so I can afford to buy my own personal theme park. I'm 

### Run inference with a test set

This is a test set of mine that I collected using the last notebook. You'll run with your own data later on in the notebook.

In [ ]:
# First upload the dataset
!wget https://raw.githubusercontent.com/n8rob/corpora/refs/heads/master/srn-eng_from_html.jsonl

--2026-07-24 15:38:38--  https://raw.githubusercontent.com/n8rob/corpora/refs/heads/master/srn-eng_from_html.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15660 (15K) [text/plain]
Saving to: ‘srn-eng_from_html.jsonl’

srn-eng_from_html.j 100%[===================>]  15.29K  --.-KB/s    in 0.001s  

2026-07-24 15:38:39 (23.6 MB/s) - ‘srn-eng_from_html.jsonl’ saved [15660/15660]



In [ ]:
# Then load data
import json

with open("srn-eng_from_html.jsonl", 'r') as f:
    bitext = json.load(f)

srns = [s for s, e in bitext]
engs = [e for s, e in bitext]

In [ ]:
# We need a couple functions to get hypotheses from the model!
def run_batch(messages):
    """
    Using the same batch code from above

    messages looks like:
    [
        [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": PROMPT1},
        ],
        [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": PROMPT},
        ],
        ...
    ]
    """
    # Process input
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # Generate output
    outputs = model.generate(**inputs, max_new_tokens=1024)
    responses = [
        processor.decode(
            output[input_len:], skip_special_tokens=True
                        ) for output in outputs
    ]

    return responses

# Template for prompts
PROMPT_TEMP = """Translate the following sentence from {src_lang} to {tgt_lang}:
{sent}

Give only the {tgt_lang} translation, without any explanatory text."""

def get_hyps(
        srcs,
        src_lang,
        tgt_lang,
        batch_size=32,
        system_temp="You are an expert translator.",
        prompt_temp=PROMPT_TEMP,
             ):
    """
    We need to:
        1. Batch inputs
        2. Put into messages format to run batch
        3. Collect responses
    """
    hyps = []
    for i in range(0, len(srcs), batch_size):
        batch_srcs = srcs[i:i+batch_size]
        messages = [
            [
                {"role": "system", "content": system_temp},
                {
                    "role": "user", "content": prompt_temp.format(
                        src_lang=src_lang,
                        tgt_lang=tgt_lang,
                        sent=sent,
                    )
                },
            ] for sent in batch_srcs
        ]
        batch_hyps = run_batch(messages)
        hyps.extend(batch_hyps)

    return hyps

### Scoring outputs

Now that we have a function to run inference, we can do so easily and compute performance scores.

We will use BLEU and chrF++, two common performance metrics for translation.

#### First Sranan Tongo $\rightarrow$ English

In [ ]:
eng_hyps = get_hyps(
    srcs=srns,
    src_lang="Sranan Tongo",
    tgt_lang="English",
)

assert len(eng_hyps) == len(srns)
print(f"Retrieved {len(eng_hyps)} hyps")

Retrieved 130 hyps


In [ ]:
from sacrebleu.metrics import BLEU, CHRF

bleu = BLEU()
chrf = CHRF(word_order=2) # word_order=2 means it's chrF++

print("BLEU score:", bleu.corpus_score(eng_hyps, [engs]).score)
print("chrF++ score:", chrf.corpus_score(eng_hyps, [engs]).score)

BLEU score: 4.532184411711639
chrF++ score: 18.794701517963915


When I ran this I got a BLEU score of $\lt 5$ and a chrF++ of $\lt 20$, which indicate _very poor_ quality. Let's look at the hyps and refs themselves.

In [ ]:
for hyp, ref in zip(eng_hyps[:5], engs[:5]):
    print(
        f"[{ref}]\n was translated as [{hyp}]"
    )
    print("-" * 10)
    print()

[When Hendrik arrived today, he didn’t greet anyone.]
 was translated as [Hendrik died, and no one was happy.]
----------

[You need to shut the window or else the rain will blow in.]
 was translated as [You must keep the fence or the water will be in danger.]
----------

[Before you pound the peanuts, you need to winnow them (lit: wave the skins away).]
 was translated as [If you stamp the paper, you will find the ink.]
----------

[Boy, don’t you have eyes? (i.e. can’t you look any better than that?)]
 was translated as [Young man, do you have any money?]
----------

[All children four years old and up have to go to school.]
 was translated as [The child is going to the store.]
----------



#### Now English $\rightarrow$ Sranan Tongo

In [ ]:
srn_hyps = get_hyps(
    srcs=engs,
    src_lang="English",
    tgt_lang="Sranan Tongo",
)

assert len(srn_hyps) == len(engs)
print(f"Retrieved {len(srn_hyps)} hyps")

Retrieved 130 hyps


In [ ]:
print("BLEU score:", bleu.corpus_score(srn_hyps, [srns]).score)
print("chrF++ score:", chrf.corpus_score(srn_hyps, [srns]).score)

BLEU score: 0.49370971645411205
chrF++ score: 13.537618617576829


This performance is even more abysmal; indicating that probaby there weren't even any common words between hypotheses and references. Let's take a look....

In [ ]:
for hyp, ref in zip(srn_hyps[:5], srns[:5]):
    print(
        f"[{ref}]\n was translated as [{hyp}]"
    )
    print("-" * 10)
    print()

[Di Hendrik doro tide, a no taki no wan sma odi.]
 was translated as [Weda Hendrik a kom today, him nuh greet nobody.]
----------

[Yu mu tapu a fensre noso a alen o wai kon inisei.]
 was translated as [Bo mester taka windo of else di rain wan blow in.]
----------

[Fosi yu stampu a pinda, yu mu wai a buba puru.]
 was translated as [Obo bo bo peanuts, bo tin mecirnan (lit: wave the skins away).]
----------

[Yongu, yu no abi ai fu si?]
 was translated as [Boy, bo no have eye?]
----------

[Ala pikin fu fo yari abi fu go na skoro.]
 was translated as [Tur tur yong (four years old) an' up hasu bo bai skool.]
----------



Sadly since I don't speak Sranan Tongo, I can't tell if these translations are any good. But they have very little overlap with the references.

It actually looks to me like the model might be generating in a completely different anglophone Creole language.

## Now run on your dataset!

First you need to upload your own data!

You can do this by:

1. Click the folder icon on the right
2. Click the upload icon and upload the file with your dataset
3. Write code in the next cell to load the data into two lists of `str` objects, named `srcs` and `tgts`.

In [ ]:
# Load the Mashi–French JSONL produced by DataExtractionAfricompling.ipynb.
import json
from urllib.request import urlopen

DATA_URL = (
    "https://raw.githubusercontent.com/Ashuza11/drc-bitext-eval/"
    "main/language_resources/mashi-shr/data/"
    "mashi-french_bitext_candidates.jsonl"
)

# The PDF candidates need manual cleanup. Start evaluation with the safer
# verse-aligned eBible subset. Set SOURCE_FILTER = None to use every source.
SOURCE_FILTER = "eBible Exodus: shr + fraLSG"
MAX_EXAMPLES = 100  # Increase carefully: model inference can be slow/costly.
EXCLUDE_REFS = {"EXO 9:2", "EXO 12:20", "EXO 15:9", "EXO 15:15", "EXO 15:21"}

with urlopen(DATA_URL) as response:
    records = [json.loads(line) for line in response if line.strip()]

if SOURCE_FILTER is not None:
    records = [row for row in records if row["source"] == SOURCE_FILTER]
records = [row for row in records if row["ref"] not in EXCLUDE_REFS]
records = records[:MAX_EXAMPLES]

srcs = [row["mashi"] for row in records]
tgts = [row["french"] for row in records]

assert srcs and len(srcs) == len(tgts)
print(f"Loaded {len(srcs)} Mashi–French pairs")

In [ ]:
src_lang = "Mashi"
tgt_lang = "French"
print(f"Evaluating {src_lang} ↔ {tgt_lang}")

### First `src` $\rightarrow$ `tgt`

In [ ]:
tgt_hyps = get_hyps(
    srcs=srcs,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

assert len(tgt_hyps) == len(srcs)
print(f"Retrieved {len(tgt_hyps)} hyps")
print()

print("Now getting scores:")
print('=' * 10)
print("BLEU score:", bleu.corpus_score(tgt_hyps, [tgts]).score)
print("chrF++ score:", chrf.corpus_score(tgt_hyps, [tgts]).score)

print("Let's look at some of the outputs:")
print('=' * 10)
for hyp, ref in zip(tgt_hyps[:5], tgts[:5]):
    print(
        f"[{ref}]\n was translated as [{hyp}]"
    )
    print("-" * 10)
    print()

### Then the opposite direction

In [ ]:
src_hyps = get_hyps(
    srcs=tgts,
    src_lang=tgt_lang,
    tgt_lang=src_lang
)

assert len(src_hyps) == len(tgts)
print(f"Retrieved {len(src_hyps)} hyps")
print()

print("Now getting scores:")
print('=' * 10)
print("BLEU score:", bleu.corpus_score(src_hyps, [srcs]).score)
print("chrF++ score:", chrf.corpus_score(src_hyps, [srcs]).score)

print("Let's look at some of the outputs:")
print('=' * 10)
for hyp, ref in zip(src_hyps[:5], srcs[:5]):
    print(
        f"[{ref}]\n was translated as [{hyp}]"
    )
    print("-" * 10)
    print()